# 📈 Notebook 01 — Predicted Revenue (Next Month Total)

**Model**: `revenue_predictor.pkl`  
**Algorithm**: Random Forest Regressor  
**Target**: Total revenue for next simulated month (Rs.)  
**Input Features**: Monthly aggregated order stats + lag features

In [ ]:
# What: Import ML and data libraries
# Why: Random Forest from sklearn, joblib for model saving, pandas for data wrangling
# Method: Standard scientific Python imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

DATA_PATH   = '../data'
MODELS_PATH = '../models'
RANDOM_STATE = 42
os.makedirs(MODELS_PATH, exist_ok=True)

# Department avg price proxy in Rs. — no price column exists, so engineered from category
DEPT_PRICE = {1:350,2:200,3:250,4:150,5:800,6:400,7:180,8:600,9:300,
              10:200,11:500,12:700,13:350,14:280,15:200,16:300,17:450,
              18:500,19:220,20:400,21:200}
print('Config loaded ✅')

In [ ]:
# What: Load orders and product-related data
# Why: Need order history, product department mappings to build revenue features
# Method: Read CSV files, report shapes for validation
orders      = pd.read_csv(f'{DATA_PATH}/orders.csv')
op_train    = pd.read_csv(f'{DATA_PATH}/order_products__train.csv')
products    = pd.read_csv(f'{DATA_PATH}/products.csv')
departments = pd.read_csv(f'{DATA_PATH}/departments.csv')

print(f'orders: {orders.shape} | op_train: {op_train.shape}')
print(f'products: {products.shape} | departments: {departments.shape}')

In [ ]:
# What: Build enriched order-item table with department and revenue columns
# Why: Revenue must be computed by joining items → products → departments
# Method: Left merge chain: op_train → products → departments → orders
products_dept = products.merge(departments, on='department_id', how='left')
order_items = op_train.merge(products_dept[['product_id','department_id']], on='product_id', how='left')
order_items = order_items.merge(
    orders[['order_id','user_id','order_number','order_dow','reordered']],
    on='order_id', how='left'
)

# Assign price and revenue using department mapping
order_items['price']   = order_items['department_id'].map(DEPT_PRICE).fillna(200)
order_items['revenue'] = order_items['price']  # 1 unit per cart item

print('Enriched order items:')
print(order_items[['order_id','department_id','price','revenue','order_number']].head())

In [ ]:
# What: Simulate monthly periods from order_number
# Why: Dataset has no real dates; order_number is the only temporal ordering
# Method: month_num = ceil(order_number / 3) maps each 3-order window to one month
order_items['month_num'] = np.ceil(order_items['order_number'] / 3).astype(int)
print('Month range:', order_items['month_num'].min(), '→', order_items['month_num'].max())
print('Unique simulated months:', order_items['month_num'].nunique())

In [ ]:
# What: Aggregate to monthly-level feature table
# Why: Model predicts next month total revenue from this month's aggregated stats
# Method: groupby month_num → sum revenue, count orders/users/items, mean reorder
monthly = order_items.groupby('month_num').agg(
    total_revenue  = ('revenue',    'sum'),
    total_orders   = ('order_id',   'nunique'),
    total_items    = ('product_id', 'count'),
    unique_users   = ('user_id',    'nunique'),
    avg_reorder    = ('reordered',  'mean'),
    avg_basket_size= ('product_id', lambda x: len(x) / x.index.get_level_values(0).nunique())
).reset_index()

print(f'Monthly table shape: {monthly.shape}')
print(monthly.head(8).to_string(index=False))

In [ ]:
# What: Add lag features (previous month values as predictors)
# Why: Next month revenue correlates strongly with prior month revenue
# Method: pandas .shift(1) to create one-period lag columns
monthly['prev_revenue']   = monthly['total_revenue'].shift(1)
monthly['prev_orders']    = monthly['total_orders'].shift(1)
monthly['prev_users']     = monthly['unique_users'].shift(1)
monthly['revenue_growth'] = monthly['total_revenue'].pct_change()  # month-over-month growth rate
monthly['items_per_order']= monthly['total_items'] / monthly['total_orders']

monthly.dropna(inplace=True)  # remove first row (no lag available)
print(f'After lag engineering: {monthly.shape}')
print(monthly[['month_num','total_revenue','prev_revenue','revenue_growth']].head())

In [ ]:
# What: Prepare feature matrix X and target vector y for training
# Why: Separate predictors from what we want to predict
# Method: Select numeric feature columns; y = current month total_revenue
FEATURES = ['total_orders','total_items','unique_users','avg_reorder',
            'items_per_order','prev_revenue','prev_orders','prev_users','revenue_growth']

X = monthly[FEATURES].values
y = monthly['total_revenue'].values

# Train/test split — keep 20% for evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# What: Train Random Forest Regressor for monthly revenue prediction
# Why: RF handles non-linear relationships and feature interactions without scaling
# Method: 300 trees, min_samples_leaf=2 for regularization, oob_score for extra validation
model = RandomForestRegressor(
    n_estimators   = 300,
    min_samples_leaf = 2,
    max_features   = 'sqrt',   # subset features per split — prevents overfitting
    oob_score      = True,     # out-of-bag score as free extra validation
    random_state   = RANDOM_STATE,
    n_jobs         = -1        # use all CPU cores
)
model.fit(X_train, y_train)
print(f'OOB R² Score: {model.oob_score_:.4f}')

In [ ]:
# What: Evaluate model on held-out test set
# Why: Check if model generalizes and is not just memorizing training data
# Method: RMSE (same units as Rs.), MAE, R² — standard regression metrics
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('=== Model Performance ===')
print(f'RMSE  : Rs. {rmse:>12,.2f}')
print(f'MAE   : Rs. {mae:>12,.2f}')
print(f'R²    : {r2:>16.4f}')

In [ ]:
# What: Cross-validation to confirm model stability
# Why: Small dataset of monthly aggregates — CV gives more reliable performance estimate
# Method: 5-fold cross-validation with negative MSE scoring
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f'Cross-Val R² scores: {cv_scores.round(4)}')
print(f'Mean CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# What: Plot actual vs predicted revenue and feature importances
# Why: Visual confirmation that model is learning meaningful patterns
# Method: Two-panel plot — scatter (actual vs pred) + bar (feature importance)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, color='darkorange', alpha=0.8, edgecolors='white', s=80)
min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5)
axes[0].set_xlabel('Actual Revenue (Rs.)')
axes[0].set_ylabel('Predicted Revenue (Rs.)')
axes[0].set_title(f'Actual vs Predicted Revenue\nR² = {r2:.4f}')

# Feature Importances
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fi.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/chart_revenue_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✅')

In [ ]:
# What: Save trained model to .pkl file
# Why: Persist model so API can load it without retraining
# Method: joblib.dump — recommended for sklearn models with large numpy arrays
joblib.dump(model, f'{MODELS_PATH}/revenue_predictor.pkl')
print('✅ Saved: models/revenue_predictor.pkl')
print(f'   File size: {os.path.getsize(f"{MODELS_PATH}/revenue_predictor.pkl") / 1024:.1f} KB')

In [ ]:
# What: Demonstrate model inference with last month's data
# Why: Confirm .pkl works and show how to use it in the API
# Method: Load saved model, pass last month features, print prediction
loaded_model = joblib.load(f'{MODELS_PATH}/revenue_predictor.pkl')
last_features = monthly[FEATURES].iloc[[-1]].values
predicted = loaded_model.predict(last_features)[0]

print('=== Sample Prediction (Next Month) ===')
print(f'📊 Predicted Total Revenue : Rs. {predicted:,.2f}')
print(f'📊 Predicted Daily Average  : Rs. {predicted/30:,.2f}')